# Лабораторная работа №3
## Вариант 3.2
#### Выполнили: Аверьянова Мария, Калягин Дмитрий, Кашникова Анна, Климович Анна


## Исходная задача

Рассмотрим задачу минимизации:

$$
\begin{aligned}
\text{minimize} \quad & f(x) = \sum_{i=1}^n x_i \log x_i \\
\text{subject to} \quad & Ax = b
\end{aligned}
$$

где $x \in \mathbb{R}^n$, $A \in \mathbb{R}^{p \times n}$, $b \in \mathbb{R}^p$, $p < n$.



## 1. Исследование на выпуклость

Целевая функция $f(x)$ является суммой функций вида $h(u) = u \log u$. Исследуем выпуклость этой функции на области $x_i > 0$:

$$
h''(u) = \frac{d^2}{du^2}(u \log u) = \frac{d}{du}(1 + \log u) = \frac{1}{u}
$$

Так как $\frac{1}{u} > 0$ для всех $u > 0$, функция $u \log u$ строго выпукла.
Сумма строго выпуклых функций также строго выпукла, поэтому $f(x)$ строго выпукла на области $x_i > 0$.

Ограничения $Ax = b$ задают аффинное множество, которое является выпуклым.

Поэтому исходная задача — выпуклая.



## Формулировка двойственной задачи

Запишем лагранжиан задачи:

$$
L(x, \lambda) = \sum_{i=1}^n x_i \log x_i + \lambda^T(Ax - b)
$$

Для нахождения двойственной функции $q(\lambda) = \inf_x L(x, \lambda)$ приравняем частную производную по каждому $x_i$ к нулю:

$$
\frac{\partial L}{\partial x_i} = \log x_i + 1 + a_i^T \lambda = 0
$$

Отсюда находим оптимальное $x^*$, выраженное через $\lambda$:

$$
\log x_i^*(\lambda) = -1 - a_i^T \lambda \quad \Rightarrow \quad x_i^*(\lambda) = \exp(-1 - a_i^T \lambda)
$$

Подставим $x^*(\lambda)$ обратно в лагранжиан:

$$
\begin{aligned}
q(\lambda) &= L(x^*(\lambda), \lambda) \\
&= \sum_{i=1}^n x_i^* \log x_i^* + \lambda^T A x^* - \lambda^T b \\
&= \sum_{i=1}^n x_i^* (-1 - a_i^T \lambda) + \sum_{i=1}^n x_i^* (a_i^T \lambda) - b^T \lambda \\
&= \sum_{i=1}^n (-x_i^*) - b^T \lambda \\
&= -\sum_{i=1}^n \exp(-1 - a_i^T \lambda) - b^T \lambda
\end{aligned}
$$

Таким образом, двойственная функция имеет вид:

$$
q(\lambda) = -b^T \lambda - \sum_{i=1}^n \exp(-1 - a_i^T \lambda)
$$

Двойственная задача заключается в максимизации этой функции:

$$
\max_{\lambda \in \mathbb{R}^p} \quad q(\lambda)
$$

Или в форме минимизации:

$$
\text{minimize} \quad b^T \lambda + \sum_{i=1}^n \exp(-1 - a_i^T \lambda)
$$

# 2. Генерация тестовых примеров и эталонное решение через CVXPY

In [ ]:
import numpy as np
import cvxpy as cp
from scipy.linalg import svd
from typing import List, Dict
import time


ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ



In [ ]:

def entropy_objective(x: np.ndarray) -> float:
    if np.any(x <= 0):
        return np.inf
    return np.sum(x * np.log(x))


def recover_primal_from_dual(nu: np.ndarray, A: np.ndarray) -> np.ndarray:
    return np.exp(-A.T @ nu - 1)


def generate_full_rank_matrix(p: int, n: int, rng):
    while True:
        A = rng.standard_normal((p, n))
        if np.linalg.matrix_rank(A) == p:
            return A


def nullspace_basis(A: np.ndarray, tol=1e-12) -> np.ndarray:
    U, S, Vt = svd(A, full_matrices=True)
    rank = np.sum(S > tol * S[0])
    return Vt[rank:].T


ГЕНЕРАЦИЯ ЗАДАЧИ

In [ ]:
def generate_test_problem(n: int, p: int, seed: int):
    rng = np.random.default_rng(seed)

    A = generate_full_rank_matrix(p, n, rng)

    x_feas = 0.5 + np.abs(rng.standard_normal(n))
    b = A @ x_feas

    N = nullspace_basis(A)

    return {
        'A': A,
        'b': b,
        'x_feas': x_feas,
        'N': N,
        'rng': rng
    }


СТАРТОВЫЕ ТОЧКИ


In [ ]:
def generate_feasible_starts(x_ref, N, num_starts, rng):
    starts = []
    n_p = N.shape[1]

    if n_p == 0:
        return [x_ref.copy() for _ in range(num_starts)]

    for _ in range(num_starts):
        z = rng.standard_normal(n_p)
        direction = N @ z

        alpha_max = np.inf
        mask = direction < 0
        if np.any(mask):
            alpha_max = np.min(x_ref[mask] / (-direction[mask])) * 0.5

        alpha = alpha_max * (0.2 + 0.6 * rng.random()) if np.isfinite(alpha_max) else 0

        x0 = x_ref + alpha * direction
        x0 = np.maximum(x0, 1e-8)

        starts.append(x0)

    return starts

CVXPY РЕШЕНИЕ

In [ ]:

def solve_primal(A, b):
    n = A.shape[1]
    x = cp.Variable(n)

    objective = cp.Minimize(cp.sum(-cp.entr(x)))
    constraints = [A @ x == b, x >= 1e-9]

    prob = cp.Problem(objective, constraints)
    prob.solve(solver=cp.SCS, eps=1e-8, max_iters=10000)

    if x.value is None:
        raise RuntimeError("Primal solve failed")

    x_star = x.value
    f_star = entropy_objective(x_star)

    return x_star, f_star


def solve_dual(A, b):
    p = A.shape[0]
    nu = cp.Variable(p)

    dual_expr = -b @ nu - cp.sum(cp.exp(-1 - A.T @ nu))

    prob = cp.Problem(cp.Maximize(dual_expr))
    prob.solve(solver=cp.SCS, eps=1e-8, max_iters=10000)

    if nu.value is None:
        raise RuntimeError("Dual solve failed")

    nu_star = nu.value
    q_star = dual_expr.value

    return nu_star, q_star

In [ ]:

def build_dataset():
    n_values = range(10, 101, 10)
    num_instances = 5
    num_starts = 5

    problems: List[Dict] = []

    for n in n_values:
        p = n // 2

        print(f"\n=== n = {n}, p = {p} ===")

        for inst_id in range(num_instances):
            seed = 1000 * n + inst_id

            prob = generate_test_problem(n, p, seed)
            A, b, N, rng = prob['A'], prob['b'], prob['N'], prob['rng']

            # CVXPY решения
            x_star, f_star = solve_primal(A, b)
            nu_star, q_star = solve_dual(A, b)

            # восстановление
            x_dual = recover_primal_from_dual(nu_star, A)

            gap = abs(f_star - q_star)

            # стартовые точки
            starts = generate_feasible_starts(x_star, N, num_starts, rng)

            print(f"Instance {inst_id}: gap = {gap:.2e}")

            problems.append({
                'n': n,
                'p': p,
                'A': A,
                'b': b,
                'N': N,
                'x_star': x_star,
                'f_star': f_star,
                'nu_star': nu_star,
                'q_star': q_star,
                'starts': starts
            })

    return problems

In [ ]:
problems = build_dataset()

import pickle
with open("entropy_problems.pkl", "wb") as f:
    pickle.dump(problems, f)

print("\nDataset saved to entropy_problems.pkl")


=== n = 10, p = 5 ===
Instance 0: gap = 9.33e-11
Instance 1: gap = 2.07e-12
Instance 2: gap = 1.64e-12
Instance 3: gap = 1.51e-11
Instance 4: gap = 3.48e-12

=== n = 20, p = 10 ===
Instance 0: gap = 3.91e-13
Instance 1: gap = 1.07e-10
Instance 2: gap = 2.43e-11
Instance 3: gap = 1.95e-12
Instance 4: gap = 1.50e-10

=== n = 30, p = 15 ===
Instance 0: gap = 1.22e-11
Instance 1: gap = 7.41e-11
Instance 2: gap = 1.93e-13
Instance 3: gap = 5.86e-12
Instance 4: gap = 1.29e-12

=== n = 40, p = 20 ===
Instance 0: gap = 4.65e-13
Instance 1: gap = 2.24e-11
Instance 2: gap = 3.86e-13
Instance 3: gap = 2.26e-12
Instance 4: gap = 4.93e-11

=== n = 50, p = 25 ===
Instance 0: gap = 7.43e-13
Instance 1: gap = 4.82e-11
Instance 2: gap = 3.91e-12
Instance 3: gap = 2.53e-11
Instance 4: gap = 1.47e-10

=== n = 60, p = 30 ===
Instance 0: gap = 5.77e-12
Instance 1: gap = 2.86e-12
Instance 2: gap = 1.44e-10
Instance 3: gap = 3.14e-12
Instance 4: gap = 8.99e-12

=== n = 70, p = 35 ===
Instance 0: gap = 1.70e

# 3. Для прямой и двойственной задач, каждого $n \in \{10, 20, \ldots, 100\}$, каждого тестового примера и каждой из 5 начальных точек реализуйте следующие методы для точности $\varepsilon = 0.01^2$:

   a) демпфированный метод Ньютона для задачи с линейными ограничениями (Boyd, §10.2, p. 525); сопоставьте результаты для прямой и двойственной задач;
   
   b) комбинированную схему, в которой на начальном этапе применяется градиентный спуск, а затем осуществляется переход к методу Ньютона;
   
   c) методы Broyden и BFGS, использующие следующие формулы обновления аппроксимации обратного гессиана:

   $$H^{\text{new}} = H + \frac{(s - Hy)s^T H}{s^T H y}$$

   для метода Broyden и

   $$H^{\text{new}} = \left(I - \frac{sy^T}{y^T s}\right) H \left(I - \frac{ys^T}{y^T s}\right) + \frac{ss^T}{y^T s}$$

   для метода BFGS, где

   $$s = x^{k+1} - x^k, \qquad y = \nabla f(x^{k+1}) - \nabla f(x^k).$$


---

$^2$Под точностью понимается выполнение неравенства $|f(x^k) - f^*| \le \varepsilon$, где $f^*$ — оптимальное значение функции.

In [ ]:
# EPS = 1e-2
# MAX_ITER = 100

# def newton_primal(problem, x0):
#     A = problem['A']
#     b = problem['b']
#     f_star = problem['f_star']

#     x = x0.copy()
#     history = []

#     for k in range(MAX_ITER):
#         grad = np.log(x) + 1
#         H = np.diag(1 / x)

#         # KKT система
#         KKT = np.block([
#             [H, A.T],
#             [A, np.zeros((A.shape[0], A.shape[0]))]
#         ])

#         rhs = -np.concatenate([grad, np.zeros(A.shape[0])])

#         sol = np.linalg.solve(KKT, rhs)
#         dx = sol[:len(x)]

#         # демпфирование
#         t = 1.0
#         while np.any(x + t * dx <= 0):
#             t *= 0.5

#         x = x + t * dx

#         f_val = np.sum(x * np.log(x))
#         history.append(f_val)

#         if abs(f_val - f_star) <= EPS:
#             return x, history, k+1

#     return x, history, MAX_ITER
EPS = 1e-2
MAX_ITER = 100

def newton_primal(problem, x0):
    A = problem['A']
    b = problem['b']
    f_star = problem['f_star']

    x = x0.copy()
    history = []

    # Параметры для line search
    alpha = 0.01
    beta = 0.5

    for k in range(MAX_ITER):
        grad = np.log(x) + 1
        H = np.diag(1 / x)
        f_val = np.sum(x * np.log(x))
        history.append(f_val)

        # 1. Проверка критерия остановки
        if abs(f_val - f_star) <= EPS:
            return x, history, k

        # 2. Решение системы KKT
        KKT = np.block([
            [H, A.T],
            [A, np.zeros((A.shape[0], A.shape[0]))]
        ])
        rhs = -np.concatenate([grad, np.zeros(A.shape[0])])

        try:
            sol = np.linalg.solve(KKT, rhs)
            dx = sol[:len(x)]
        except np.linalg.LinAlgError:
            print("KKT matrix singular")
            break

        # 3. Демпфирование (Backtracking line search)
        t = 1.0
        while True:
            x_new = x + t * dx
            # Проверка 1: Допустимость (x > 0)
            if np.any(x_new <= 0):
                t *= beta
                continue
            # Проверка 2: Убывание функции (Armijo condition)
            f_new = np.sum(x_new * np.log(x_new))
            if f_new <= f_val + alpha * t * (grad @ dx):
                break

            t *= beta


        x = x_new

    return x, history, MAX_ITER

ДВОЙСТВЕННАЯ ЗАДАЧА: NEWTON

In [ ]:
def dual_objective(nu, A, b):
    return -b @ nu - np.sum(np.exp(-1 - A.T @ nu))


def dual_grad(nu, A, b):
    exp_term = np.exp(-1 - A.T @ nu)
    return -b + A @ exp_term


def dual_hess(nu, A):
    exp_term = np.exp(-1 - A.T @ nu)
    return A @ np.diag(exp_term) @ A.T


# def newton_dual(problem, nu0):
#     A = problem['A']
#     b = problem['b']
#     q_star = problem['q_star']

#     nu = nu0.copy()
#     history = []

#     for k in range(MAX_ITER):
#         grad = dual_grad(nu, A, b)
#         H = dual_hess(nu, A)

#         dnu = np.linalg.solve(H, grad)

#         # line search
#         t = 1.0
#         while True:
#             nu_new = nu + t * dnu
#             if dual_objective(nu_new, A, b) >= dual_objective(nu, A, b):
#                 break
#             t *= 0.5

#         nu = nu + t * dnu

#         q_val = dual_objective(nu, A, b)
#         history.append(q_val)

#         if abs(q_val - q_star) <= EPS:
#             return nu, history, k+1

#     return nu, history, MAX_ITER


def newton_dual(problem, nu0):
    A = problem['A']
    b = problem['b']
    q_star = problem['q_star']

    nu = nu0.copy()
    history = []

    alpha = 0.01
    beta = 0.5

    for k in range(MAX_ITER):
        grad = dual_grad(nu, A, b)
        H = dual_hess(nu, A)
        q_val = dual_objective(nu, A, b)
        history.append(q_val)

        if abs(q_val - q_star) <= EPS:
            return nu, history, k

        # Шаг Ньютона для максимизации: dnu = H^{-1} * grad
        # (так как реальный гессиан -H, а шаг = -(-H)^{-1}grad)
        try:
            dnu = np.linalg.solve(H, grad)
        except np.linalg.LinAlgError:
            break

        # Line search для максимизации
        t = 1.0
        while True:
            nu_new = nu + t * dnu
            q_new = dual_objective(nu_new, A, b)
            # Проверяем рост функции (Armijo для максимизации)
            if q_new >= q_val + alpha * t * (grad @ dnu):
                break
            t *= beta
            if t < 1e-12:
                break

        nu = nu_new

    return nu, history, MAX_ITER

In [ ]:
def hybrid_method(problem, x0):
    raise NotImplementedError("Hybrid method not implemented")



def broyden_method(problem, x0):
    raise NotImplementedError("Broyden not implemented")


def bfgs_method(problem, x0):
    raise NotImplementedError("BFGS not implemented")

СРАВНЕНИЕ МЕТОДОВ

In [ ]:
def run_experiments(problems):
    results = []

    for prob_id, problem in enumerate(problems):
        A = problem['A']
        p = A.shape[0]

        for start_id, x0 in enumerate(problem['starts']):

            # старт для двойственной
            nu0 = np.zeros(p)

            # ---- PRIMAL ----
            t0 = time.time()
            x_sol, hist_p, it_p = newton_primal(problem, x0)
            time_p = time.time() - t0

            # ---- DUAL ----
            t0 = time.time()
            nu_sol, hist_d, it_d = newton_dual(problem, nu0)
            time_d = time.time() - t0

            results.append({
                'problem_id': prob_id,
                'n': problem['n'],
                'start_id': start_id,

                'primal_iters': it_p,
                'dual_iters': it_d,

                'primal_time': time_p,
                'dual_time': time_d,

                'primal_final_error': abs(hist_p[-1] - problem['f_star']),
                'dual_final_error': abs(hist_d[-1] - problem['q_star'])
            })

            print(f"n={problem['n']} | start={start_id} | "
                  f"primal it={it_p}, dual it={it_d}")

    return results

In [ ]:
results = run_experiments(problems)

n=10 | start=0 | primal it=1, dual it=3
n=10 | start=1 | primal it=0, dual it=3
n=10 | start=2 | primal it=1, dual it=3
n=10 | start=3 | primal it=2, dual it=3
n=10 | start=4 | primal it=1, dual it=3
n=10 | start=0 | primal it=0, dual it=3
n=10 | start=1 | primal it=1, dual it=3
n=10 | start=2 | primal it=1, dual it=3
n=10 | start=3 | primal it=0, dual it=3
n=10 | start=4 | primal it=2, dual it=3
n=10 | start=0 | primal it=2, dual it=3
n=10 | start=1 | primal it=0, dual it=3
n=10 | start=2 | primal it=1, dual it=3
n=10 | start=3 | primal it=1, dual it=3
n=10 | start=4 | primal it=1, dual it=3
n=10 | start=0 | primal it=1, dual it=2
n=10 | start=1 | primal it=2, dual it=2
n=10 | start=2 | primal it=1, dual it=2
n=10 | start=3 | primal it=1, dual it=2
n=10 | start=4 | primal it=1, dual it=2
n=10 | start=0 | primal it=2, dual it=2
n=10 | start=1 | primal it=0, dual it=2
n=10 | start=2 | primal it=0, dual it=2
n=10 | start=3 | primal it=2, dual it=2
n=10 | start=4 | primal it=1, dual it=2
